### reservoir

In [13]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score

# --- Load data ---
rows = []
with open("reservoir_data.txt") as f:
    for line in f:
        line = line.strip()
        if not line or line == "DONE":
            continue
        vals = [float(x) for x in line.split(",")]
        if len(vals) == 51:   # 1 input + 50 states
            rows.append(vals)

data   = np.array(rows)
rows = []
with open("reservoir_data.txt") as f:
    for line in f:
        line = line.strip()
        if not line or line == "DONE":
            continue
        vals = [float(x) for x in line.split(",")]
        if len(vals) == 41:   # 1 input + 40 states
            rows.append(vals)
        else:
            print(f"Skipping line with {len(vals)} values")

if len(rows) == 0:
    raise ValueError("No valid rows loaded from reservoir_data.txt")

data = np.array(rows, dtype=float)
inputs = data[:, 0].astype(int)
states = data[:, 1:]               # columns 1-50 = reservoir state

# --- XOR task: output[t] = input[t] XOR input[t-1] ---
targets = np.array([inputs[t] ^ inputs[t-1] for t in range(len(inputs))])

# Discard t=0 (no previous input yet), skip first 20 as washout
washout = 20
X = states[washout:]
y = targets[washout:]

split = int(len(X) * 0.7)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# --- Train linear readout ---
model = Ridge(alpha=1e-3)
model.fit(X_train, y_train)

y_pred = (model.predict(X_test) > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)
print(f"XOR accuracy: {acc:.1%}  (chance = 50%)")

# --- Sanity: how different are states for input=0 vs input=1? ---
s0 = states[washout:][inputs[washout:] == 0].mean(axis=0)
s1 = states[washout:][inputs[washout:] == 1].mean(axis=0)
print(f"Mean state difference (0 vs 1): {np.abs(s1-s0).mean():.1f} counts")



XOR accuracy: 99.3%  (chance = 50%)
Mean state difference (0 vs 1): 315.0 counts


In [14]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(Ridge(alpha=1e-3), X, y, cv=5)
print(f"5-fold CV accuracy: {scores.mean():.1%} ± {scores.std():.1%}")

5-fold CV accuracy: 91.6% ± 3.3%


In [ ]:
# NARMA-2: y(t) = 0.4*y(t-1) + 0.4*y(t-1)*y(t-2) + 0.6*u(t)**3 + 0.1
# y(t)=0.4y(t−1)+0.4y(t−1)y(t−2)+0.6u(t)u(t−1)+0.1
u = inputs.astype(float) * 0.5   # scale {0,1} → {0, 0.5}

narma = np.zeros(len(u))
for t in range(2, len(u)):
    narma[t] = (0.4*narma[t-1] 
                + 0.4*narma[t-1]*narma[t-2] 
                + 0.6*u[t]*u[t-1]           # <-- multiply, not cube
                + 0.1)

# Sanity check - should be small finite numbers
print(f"NARMA range: {narma.min():.3f} to {narma.max():.3f}")  

X2 = states[washout:]
y2 = narma[washout:]
split2 = int(len(X2) * 0.7)

model2 = Ridge(alpha=1e-3)
model2.fit(X2[:split2], y2[:split2])
y2_pred = model2.predict(X2[split2:])
nmse = np.mean((y2[split2:] - y2_pred)**2) / np.var(y2[split2:])
print(f"NARMA-2 NMSE: {nmse:.4f}  (< 0.1 is good, < 0.05 is excellent)")

NARMA range: 0.000 to 0.726
NARMA-2 NMSE: 0.4472  (< 0.1 is good, < 0.05 is excellent)


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X2_train = scaler.fit_transform(X2[:split2])
X2_test  = scaler.transform(X2[split2:])

model2 = Ridge(alpha=1e-3)
model2.fit(X2_train, y2[:split2])
y2_pred = model2.predict(X2_test)
nmse = np.mean((y2[split2:] - y2_pred)**2) / np.var(y2[split2:])
print(f"NARMA-2 NMSE (normalized): {nmse:.4f}")

NARMA-2 NMSE (normalized): 0.4469


### reservoir 2 (NARMA 2)

In [18]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score

# --- Load data ---
rows = []
with open("resrvoir1.txt") as f:
    for line in f:
        line = line.strip()
        if not line or line == "DONE":
            continue
        vals = [float(x) for x in line.split(",")]
        if len(vals) == 51:   # 1 input + 50 states
            rows.append(vals)

data   = np.array(rows)
inputs = data[:, 0].astype(float)   # column 0 = input bit
states = data[:, 1:]               # columns 1-50 = reservoir state

# # --- XOR task: output[t] = input[t] XOR input[t-1] ---
# targets = np.array([inputs[t] ^ inputs[t-1] for t in range(len(inputs))])

# # Discard t=0 (no previous input yet), skip first 20 as washout
# washout = 20
# X = states[washout:]
# y = targets[washout:]

# split = int(len(X) * 0.7)
# X_train, X_test = X[:split], X[split:]
# y_train, y_test = y[:split], y[split:]

# # --- Train linear readout ---
# model = Ridge(alpha=1e-3)
# model.fit(X_train, y_train)

# y_pred = (model.predict(X_test) > 0.5).astype(int)
# acc = accuracy_score(y_test, y_pred)
# print(f"XOR accuracy: {acc:.1%}  (chance = 50%)")

# # --- Sanity: how different are states for input=0 vs input=1? ---
# s0 = states[inputs[washout:] == 0].mean(axis=0)
# s1 = states[inputs[washout:] == 1].mean(axis=0)
# print(f"Mean state difference (0 vs 1): {np.abs(s1-s0).mean():.1f} counts")

In [ ]:
# NARMA-2: y(t) = 0.4*y(t-1) + 0.4*y(t-1)*y(t-2) + 0.6*u(t)**3 + 0.1
u = inputs   # scale {0,1} → {0, 0.5}

narma = np.zeros(len(u))
for t in range(2, len(u)):
    narma[t] = (0.3*narma[t-1]             # was 0.4
            + 0.05*narma[t-1]*narma[t-2]   
            + 0.1*u[t-1]               # adjust input term too
            + 0.1)

# Sanity check - should be small finite numbers
print(f"NARMA range: {narma.min():.3f} to {narma.max():.3f}")  

X2 = states[washout:]
y2 = narma[washout:]
split2 = int(len(X2) * 0.7)

model2 = Ridge(alpha=1e-3)
model2.fit(X2[:split2], np.nan_to_num(y2[:split2], nan=0.0, posinf=0.0, neginf=0.0))
y2_pred = model2.predict(X2[split2:])
nmse = np.mean((y2[split2:] - y2_pred)**2) / np.var(y2[split2:])
print(f"NARMA-2 NMSE: {nmse:.4f}  (< 0.1 is good, < 0.05 is excellent)")

NARMA range: 0.000 to 0.288
NARMA-2 NMSE: 0.1820  (< 0.1 is good, < 0.05 is excellent)
